## Notion of _lvalue_ and _rvalue_
- An object / expression is an  _rvalue_ if it can generate a value -- usually literals / constants / identifiers / variables are rvalues
- An object / expression is an  _lvalue_ if it can receive a value -- usually identifiers / variables are lvalues
    - Constants / literals are not allowed to be lvalues in Python
    - `a = 4` works but `4 = a` will cause an error
    - Certain pure functional programmming languages allow such inversions but not Python
- Tuple / list unpacking can be both an as well as an rvalue and an lvalue
- The unpacking operator is somewhat adaptive
    - Packed and non-packed variables may be used together in an assignment
    - All variables will get a value with packed variable possibly getting more than one value
    - Value order preserved while assigning them to variables
    - **Warning**: Only one packed variable allowed since otherwise the assignment is ambiguous

In [1]:
my_tuple = list( range( 10 ) )
( a, b, *c ) = my_tuple
print( "a = ", a, "b = ", b, "c = ", c )
( a, *b, c ) = my_tuple
print( "a = ", a, "b = ", b, "c = ", c )
( a, *b, *c ) = my_tuple # Will generate an error since this is an ambiguous case
print( "a = ", a, "b = ", b, "c = ", c )

a =  0 b =  1 c =  [2, 3, 4, 5, 6, 7, 8, 9]
a =  0 b =  [1, 2, 3, 4, 5, 6, 7, 8] c =  9


SyntaxError: multiple starred expressions in assignment (3571654178.py, line 6)

# Dictionary comprehension with conditional values
- Dictionary comprehension is of the form `[ c_expr : c_expr for < iteration expr here > if < conditional expr here > ]`
    - `c_expr` is a "_conditional expressions_" of the form ` < val1 > if < condition1 > else < val2 > if < condition2 > else < val3> `
    - Python expects two conditional expressions, one for key, one for value
        - Thus, the key expression can have its own if-else going on
        - The value expression can have its own if-else going on
        - These expressions must always be "productive"
        - This is why must have a catch-all `else` clause
    - Note that there can even be a separate "filter" clause at the very end
    - Not the most intuitive design -- PEP308 seems to be behind this
- **Rabbit Hole**: https://docs.python.org/3/reference/expressions.html#dictionary-displays

### Gives us a way to solve the neighbor_map problem in a single dictionary comprehension ( no dict update )

In [2]:
txt = "it is raining today abab"
alphabet = list( map( chr, range( ord( 'a' ), ord( 'z' ) + 1 ) ) )
neighbors = list( map( lambda c: { n for n in alphabet if ( c + n in txt ) and ( not n == ' ' ) }, alphabet ) )
d_neighbors = { c : ( n if c in txt else { None } ) for c, n in zip( alphabet, neighbors ) }
print( d_neighbors )

{'a': {'i', 'y', 'b'}, 'b': {'a'}, 'c': {None}, 'd': {'a'}, 'e': {None}, 'f': {None}, 'g': set(), 'h': {None}, 'i': {'n', 't', 's'}, 'j': {None}, 'k': {None}, 'l': {None}, 'm': {None}, 'n': {'i', 'g'}, 'o': {'d'}, 'p': {None}, 'q': {None}, 'r': {'a'}, 's': set(), 't': {'o'}, 'u': {None}, 'v': {None}, 'w': {None}, 'x': {None}, 'y': set(), 'z': {None}}


### Gives us a way to solve the first occurrence problem without currying

In [3]:
first_occurence = list( map( lambda c : txt.index( c ) if c in txt else None, alphabet ) )
print( first_occurence )

[7, 21, None, 16, None, None, 12, None, 0, None, None, None, None, 9, 15, None, None, 6, 4, 1, None, None, None, None, 18, None]


# Recursion
- A powerful computing concept
- The recursive call could be the first statement in function body ( head-recursion ) or the last ( tail-recursion )
    - Tail recursion allows processing in the direction of recursive descent
    - Head recursion allows processing in the reverse direction of recursive descent
    - There is also tree recursion where there are multiple recursive calls
        - For example merge-sort / quicksort naturally have a tree-recursive definition
    - Functional programming languages e.g. Haskell, F# good at efficient ( tail ) recursion  -- not so much with Python
- Functions calling themselves ( self-recursion ) or each other ( indirect or mutual recursion )
    - For example, Collatz rule can be defined using indirect recursion
    - Two functions -- `halve()` and `double-plus-one()` calling each other or themselves

In [4]:
# Tail recursion
def print_digits_tail( n ):
    if n < 10:
        print( n )
        return
    print( n % 10, end = '' )
    print_digits_tail( n // 10 )

print_digits_tail( 2 ** 10 )

# Head recursion
def print_digits_head( n ):
    if n < 10:
        print( n, end = '' )
        return
    print_digits_head( n // 10 )
    print( n % 10, end = '' )

print_digits_head( 2 ** 10 )
print()

# Tail recursion can usually be rewritten to use an accumulator
# The accumulator stores progress in previous calls and progress made so far in this call
# This is possible since in tail recursion, recursive call is at the end
# No more ( substantial ) work to be done after recursive call
def print_digits_tail_acc( n, acc ):
    if n == 0:
        print( acc )
        return
    acc += str( n % 10 )
    print_digits_tail_acc( n // 10, acc )

print_digits_tail_acc( 2 ** 10, "" )

# Tail recursion is usually simpler to proceduralize using loops
# Creating an accumulator is the first step in translating recursion to procedural
acc = ""
n = 2 ** 10
while n > 0:
    acc += str( n % 10 )
    n = n // 10
print( acc )

4201
1024
4201
4201


# Recursion is expensive
- Python limits the depth of recursion
- For my system it is 3000
- Can be increased but unwise to do so

In [5]:
# This limit can be reset
import sys
sys.getrecursionlimit()

3000

# Recursion is expensive
- Notion of **Call Stack** 
- Recursive solutions are often conceptually simple but resource-hungry
- Better to translate recursive code to procedural code
    - Tail recursion is usually easier to _proceduralize_ since it usually allows result accumulation
- **Church-Turing Thesis** $\Rightarrow$ for every recursive solution, there exists a procedural one too!

In [6]:
def fib( n, stack ):
    # Print the call stack
    print( stack )
    if n < 2: return 1
    # Recursive calls have a higher stack depth
    return fib( n - 1, stack + f"[{n-1}]" ) + fib( n - 2, stack + f"[{n-2}]" )

# Proceduralized version of the above recursive code
n = 5
fib_n = [ 0 for i in range( n + 1 ) ]
fib_n[ 0 ] = 1
fib_n[ 1 ] = 1
for i in range( 2, n + 1 ):
    fib_n[ i ] = fib_n[ i - 1 ] + fib_n[ i - 2 ]

print( f"fib({n}) =", fib( n, f"[{n}]" ) )
print( f"fib({n}) =", fib_n[ -1 ] )

[5]
[5][4]
[5][4][3]
[5][4][3][2]
[5][4][3][2][1]
[5][4][3][2][0]
[5][4][3][1]
[5][4][2]
[5][4][2][1]
[5][4][2][0]
[5][3]
[5][3][2]
[5][3][2][1]
[5][3][2][0]
[5][3][1]
fib(5) = 8
fib(5) = 8


# Scope
- Scope is like a house / enclosure where things may be named differently
    - An object may be passed as an argument to a function
    - Outside the function it may be called by a different name
    - The function may call that same object a different name
- Python differs from C / C++ / Java in scoping rules
    - In those languages, even blocks ( loops, conditionals ) created a new scope
    - In Python that is not the case -- only functions, comprehensions, classes ( and some others ) create scope
- Global variables are declared outside any function / comprehension / class
    - They are visible inside every function / comprehension / class unless they are _shadowed_
- If a global variable is used as an rvalue inside a function / comprehension / class, it refers to global variable
- If a global variable is used as an lvalue inside a function / comprehension / class, it creates a local variable and shadows the global one
- To use global variable as an lvalue inside a function / comprehension / class, one must use the `global` keyword
    - It tells Python to not shadow the global variable if that variable is used in an assignment operation later on in the function etc
    - This is a quirk of Python since Python allows creation of variables simply by assigning them a value
- Python also uses the `nonlocal` keyword to refer to _parent_ scope
    - This is used to refer to variables defined ( or passed as arguments to ) the enclosing function of a function
        - Parent variables available to child function unless shadowed by a local variable of the child
        - Assigning value to a variable automatically creates a new local variable causing a shadow
        - To avoid this, and use parent variables as lvalue, use the `nonlocal` keyword

In [7]:
gg = 100

def func( num ):
    global gg
    gg = 10 # Global variable
    print( gg )

def func2( num ):
    gg = 42 # Local variable
    print( gg )

global gg
print( gg )
func( 1000 )
func2( 1000 )
print( gg )

100
10
42
10


# Functional Programming
- Python offers functional programming tools
    - Functions are objects -- can be passed as arguments, returned as values
    - Python supports _currying_ -- simplifying multi-argument functions into a sequence of single-argument functions
        - Named after the PL researcher Haskell Brooks Curry
        - Pythonic usage often means _partial application_ : fix any subset arguments, leave others unbound ( much to the chagrin of PL purists )
        - Strict currying involves creating only single-argument function and that too in a particular order
        - For example `f( a, b, c )` would be converted into three curried functions ( strictly speaking )
            - We create a function `p` of a single variable `a` that will return a function `q`
            - We create a function `q` of a single variable `b` that will return a function `r`
            - We create a function `r` of a single variable `c` that will return the value `f( a, b, c )`
            - Calling `f( a, b, c )` is equivalent to calling `p( a )( b )( c )`
        - However, in Python, we can violate this strict order
            - We create a function `m` of a two variables `a`, `c` that will return a function `n`
            - We create a function `n` of a single variable `b` that will return the value `f( a, b, c )`
            - Calling `f( a, b, c )` is equivalent to calling `m( a, c )( b )`
            - Can you see why this annoys PL purists?
    - Python supports _closures_ -- a function remembers its creation environment even after creation is done
        - More than just syntactic sugar -- a powerful programming concept
        - Very useful in event-driven programming for apps, callbacks in deep learning
    - Python supports anonymous functions ( _lambda functions_ )
        - Usually syntactic sugar to avoid creating pointless names in instances such as `map`, `filter`, `sorted` and while currying

In [8]:
def generate_counter( step ):
    count = 0
    def counter():
        # The statement count += step causes Python to think count is a local variable
        # Must use the nonlocal keyword since we do use count as an lvalue
        # Using the nonlocal keyword will tell Python to refer to the parent variable
        nonlocal count
        print( count )
        count += step
    return counter

count_by_5 = generate_counter( 5 )
count_by_5()
count_by_5()
count_by_5()

0
5
10


### I really really want to shadow the print function
- Using closures, we can overcome shadowing to some extent

In [9]:
# If you run this cell twice, it will destroy the closure and print will be shadowed once more (why??)
# Do not worry -- there is a way out -- see the cell below
def my_print( *objects, sep = ' ', end = '\n', file = None, flush = False, func = print ):
    func( *objects, sep = ' ', end = '\n', file = None, flush = False )

backup = print
print = 10
my_print( "hello", "world" )

hello world


# Python builtins
- Python offers the `__builtins__` module
    - a set of super helpful functions, constants and exceptions
    - See https://docs.python.org/3/library/functions.html#built-in-functions
- Shadowing built-ins is not a good idea
    - Can permanently disable the Python kernel requiring kernel reboot
- Python has impressive meta-programming features 
    - `eval` : evaluate single expression
    - `exec` : execute a block of statements
    - Can introduce security vulnerabilities if used in native / desktop / web apps
    - Risk of injection attacks on your app / server

In [13]:
# Even if you end up shadowing print ( say if you do run the above code twice ), there is still another way
# Can access print in a roundabout way by referring to the builtins
# What will happen if you shadow the builtins themselves???
__builtins__.print( "hello world" )

# Metaprogramming
my_print( eval( " sum( range( 10 ) ) " ) )

# This has side effects
# This will inject these new variables into your environment
exec( '''
injected_bb = 1010
injected_cc = 2020
''' )
my_print( injected_bb, injected_cc )

# Can allow access to file system
eval( "my_print( __import__( 'os' ).listdir() )" )

hello world
45
1010 2020
['.ipynb_checkpoints', 'bck_22_08_2026.ipynb', 'ChatGPT Image Jul 31, 2026, 01_42_59 AM.png', 'ChatGPT Image Jul 31, 2026, 01_50_09 AM.png', 'cursor', 'lec1.pptx', 'lec2.ipynb', 'lec3.ipynb', 'lec4.ipynb', 'lec5.ipynb', 'lec6.ipynb', 'new 1.py', 'ps1.docx', 'ps1.pdf', 'ps2.docx', 'ps2.pdf', 'ps3.docx', 'ps3.pdf', 'Quiz1_judge.ipynb', 'tut1.ipynb', 'tut2.ipynb', 'tut3.ipynb']


# File IO
- `open` : open and read / write files
    - File properties : `name`, `mode`, `closed`
    - _Basic Modes_ : `r` ( read only ), `w` ( erase and write only ),`a` ( append ), `x` ( create only )
    - _Extended Modes_ : `r+` ( read-write ),  `w+` ( erase and read-write ), `a+` ( read and append ), 
    - **Warning**: opening a file in `w` or `w+` modes will immediately erase its contents ( truncate it ) irrecoverably
        - The `a` and `a+` modes are safer -- do not truncate the file
    - **Note**: Opening a file in `x` mode will raise an Error if the file already exists
    - `read()`, `readlines()`, `write()`, `writelines()`
    - Must close every open file else may cause file corruption, memory leaks
    - **Contextual closing** using **execution wrappers** is the Pythonic way to avoid this
        - Syntax `with open( < file_name_or_path >, < mode > ) as < file_handle_variable >:`
        - **Note**: the syntax of contextual closing reverses the order of lvalue and rvalue
        - rvalue comes first and lvalue comes later
        - This is because of use of the `as` keyword which flips the order of lvalue and rvalue
        - The `as` keyword is also used in `import` statements
    - Assures closure even in the event of exceptions
- There are far more sophisticated ways to access specific file types ( numerical, xlsx, csv, sql )
    - Require special libraries -- will study them later

In [15]:
try:
    file_handle = open( "my_file.txt", 'r' )
except FileNotFoundError as e:
    my_print( "File not found with error: ", e )
else:
    my_print( file_handle.read() )
finally:
    try:
        file_handle
    except NameError as e:
        my_print( "File handle not created. Got error: ", e )
    else:
        file_handle.close()
    finally:
        my_print( "done recovering" )

File not found with error:  [Errno 2] No such file or directory: 'my_file.txt'
File handle not created. Got error:  name 'file_handle' is not defined
done recovering


# Exceptions and Errors
- Not much difference -- in fact, some errors are secretly exceptions
- They signal either improper input or improper computation
- Can create new types of Exceptions and `raise` them ( will see later when we study classes )
- Can be "caught" by using the keyword `except`
    - The exception / error object can be captured in an object using the `as` keyword
    - Recall that `as` flips the order of lvalue and rvalue
- If we suspect some code may cause errors or exceptions, use the `try`-`except`-`else`-`finally` block
    - The `try` block contains the code that is suspected to raise that exception
    - The `except` block tells us what to do if that exception is indeed raised
    - The `else` block ( optional ) tells us what to do if the exception is not raised
    - The `finally` block ( optional ) is always executed, whether exception is raised or not

In [23]:
# Create a new file
try:
    file_handle = open( "my_file.txt", 'x' )
except FileExistsError as e:
    my_print( "File already exists. Received error message: ", e )
finally:
    file_handle.close()

with open( "my_file.txt", 'a' ) as file_handle:
    file_handle.write( '''
Hello World
This is a file
This has several lines
Have a nice evening
done recovering
''' )

with open( "my_file.txt", 'r' ) as file_handle:
    my_print( *file_handle.readlines() )

File already exists. Received error message:  [Errno 17] File exists: 'my_file.txt'

 Hello World
 This is a file
 This has several lines
 Have a nice evening
 done recovering

